# Lesson 1: Your first subject

**You will learn:** what a *subject*, a *channel*, an *event* and a *session* are, and how they fit together.
**Requires:** CPU only, no downloads, under a minute.

In Brain-Score, a model is treated like a participant in a lab experiment. The experiment presents inputs, the participant responds, and the responses are compared with data from people or animals. The participant is called a **subject**.

```text
experiment  --inputs-->  subject  --responses-->  experiment
            <-- one session: next_input() / respond() -->
```

## Events and channels

Every input and every response is a `StreamEvent`. An event has four fields:

- `channel`: what kind of information it carries, such as `text`, `vision` or `behavior`.
- `payload`: the data itself, such as a string or an image array.
- `t_ms`: when it happens, in milliseconds (or `None` when time does not matter).
- `meta`: an optional dictionary of extra information.

Channel names come from a shared catalog, so an experiment and a subject agree on what each name means.

In [1]:
from brainscore_core.streaming import StreamEvent

word = StreamEvent(channel='text', payload='elephant', t_ms=0)
word

StreamEvent(channel='text', payload='elephant', t_ms=0, meta={})

## A session delivers inputs and collects responses

A **session** is one run of an experiment with one subject. It has two methods:

- `session.next_input()` returns the next input event, or `None` when there are no more inputs.
- `session.respond(event)` hands a response back to the experiment.

`BufferedSession` is the simplest session: it holds a fixed list of inputs and stores every response. The inputs do not have to arrive one at a time; you can put a whole batch in one event, and the subject decides how to process it.

In [2]:
from brainscore_core.streaming import BufferedSession

words = ['cat', 'elephant', 'it', 'photosynthesis']
session = BufferedSession(inputs=[StreamEvent('text', w, t_ms=i * 500) for i, w in enumerate(words)])
session.next_input()

StreamEvent(channel='text', payload='cat', t_ms=0, meta={})

## A subject declares its channels and implements `participate`

A subject is a Python class with four parts:

| Part | Meaning |
| --- | --- |
| `identifier` | A unique name. |
| `in_channels` | The input channels it can read. |
| `out_channels` | The output channels it can produce. |
| `participate(session)` | Read inputs until `next_input()` returns `None`, responding to each. |

The subject below "reads" a word and reports a reading time. It is a toy rule, not a model of reading: longer words take longer.

In [3]:
from brainscore_core import Subject

class WordLengthReader(Subject):
    identifier = 'word-length-reader'
    in_channels = {'text'}
    out_channels = {'behavior'}

    def participate(self, session):
        while (event := session.next_input()) is not None:
            reading_time_ms = 200 + 30 * len(event.payload)
            session.respond(StreamEvent('behavior', reading_time_ms, t_ms=event.t_ms))

## Run the session

Create a fresh session, let the subject participate, and collect the responses.

In [4]:
session = BufferedSession(inputs=[StreamEvent('text', w, t_ms=i * 500) for i, w in enumerate(words)])
WordLengthReader().participate(session)

for word, response in zip(words, session.collect()):
    print(f'{word:>15}: {response.payload} ms')

            cat: 290 ms
       elephant: 440 ms
             it: 260 ms
 photosynthesis: 620 ms


## Summary

- A **subject** is anything that takes part in an experiment: a neural network, a robot policy, a remote API, or a hand-written rule like this one.
- It declares which **channels** it reads and writes, and implements one method, `participate(session)`.
- A **session** supplies input **events** and collects response events.

The experiment never needs to know what is inside the subject. That is what lets the same experiment run on very different models.

**Next:** [Lesson 2](02_your_first_benchmark.ipynb) turns this exchange into a scored test.